# Credit Card Default Classification in Python

This notebook connects the reusable modules in `src/` to run the full machine-learning workflow:

1. Load and validate the dataset  
2. Clean and recode variables  
3. Create a stratified train/test split  
4. Train L1 Logistic Regression, Bagging, Random Forest, and Gradient Boosting models  
5. Compare Accuracy, Recall, Specificity, Precision, and F1 Score


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

# Make the project src/ directory importable whether the notebook is
# launched from the repository root or from notebooks/.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from preprocessing import load_data, clean_data, split_data
from modeling import build_models, train_models
from evaluation import compare_models, get_confusion_matrix


## 1. Load the dataset

The project expects the public UCI credit-card dataset to be stored at `data/UCI_Credit_Card.csv`.


In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "UCI_Credit_Card.csv"

raw_data = load_data(DATA_PATH)
print(f"Dataset shape: {raw_data.shape}")
raw_data.head()


## 2. Inspect the original outcome distribution


In [ ]:
target_raw = "default.payment.next.month"
class_counts = raw_data[target_raw].value_counts().sort_index()
class_proportions = raw_data[target_raw].value_counts(normalize=True).sort_index()

pd.DataFrame({
    "Count": class_counts,
    "Proportion": class_proportions.round(4),
})


## 3. Clean and prepare the data

The reusable preprocessing module removes the identifier, renames the target, and recodes demographic categorical variables.


In [ ]:
cleaned_data = clean_data(raw_data)
print(f"Cleaned dataset shape: {cleaned_data.shape}")
cleaned_data.head()


## 4. Create a stratified train/test split

A 70/30 split is used. Stratification preserves the proportion of default and non-default cases in both subsets.


In [ ]:
X_train, X_test, y_train, y_test = split_data(cleaned_data)

print("Training predictors:", X_train.shape)
print("Test predictors:", X_test.shape)
print("\nTraining outcome proportions:")
print(y_train.value_counts(normalize=True).sort_index().round(4))
print("\nTest outcome proportions:")
print(y_test.value_counts(normalize=True).sort_index().round(4))


## 5. Build and train the models

Each model is wrapped in a scikit-learn pipeline so preprocessing is applied consistently during fitting and prediction.


In [ ]:
models = build_models(X_train)
trained_models = train_models(models, X_train, y_train)

list(trained_models.keys())


## 6. Compare model performance

Accuracy is reported, but Recall and F1 are especially important because the default class is less common than the non-default class.


In [ ]:
comparison = compare_models(trained_models, X_test, y_test)
comparison


## 7. Visualize the model comparison


In [ ]:
plot_data = comparison.set_index("Model")[["Accuracy", "Recall", "Precision", "F1"]]

ax = plot_data.plot(kind="bar", figsize=(10, 6))
ax.set_title("Model Performance Comparison")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.set_xlabel("")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()


## 8. Inspect confusion matrices


In [ ]:
for model_name, model in trained_models.items():
    matrix = get_confusion_matrix(model, X_test, y_test)
    matrix_df = pd.DataFrame(
        matrix,
        index=["Actual No Default", "Actual Default"],
        columns=["Predicted No Default", "Predicted Default"],
    )
    print(f"\n{model_name}")
    display(matrix_df)


## 9. Interpretation

After running the notebook, summarize the practical trade-offs among the models. In particular, compare Recall and F1 rather than selecting a model only by Accuracy.
